# 8.6. Residual Networks (ResNet) and ResNeXt
D2L의 Residual Networks (ResNet) and ResNeXt장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 더 깊은 신경망이 항상 더 좋나?

지금까지 CNN은 점점 깊어졌다.

    LeNet -> AlexNet -> VGG -> GoogLeNet -> 더 깊은 네트워크  
    층을 더 많이 추가 -> 더 복잡한 특징 학습 가능 -> 성능 향상

이런식으로 갈 것 같지만 실제로는 그렇지 않다. 단순히 layer를 계속 추가한다고 기존 모델보다 항상 좋은 모델이 되는 것은 아니다.

새로운 layer를 추가했을 때, 최소한 기존 모델이 하던 것은 그대로 할 수 있어야 하지 않을까? 라는 문제에서 ResNet이 시작된다.

## 2. Identity Function이라는 아이디어

어떤 기존 네트워크가 $f(x)$ 를 잘 학습하고 있다고 할때 여기에 새로운 layer를 추가했는데 이 layer가 아무것도 바꾸지 않고

$$
x \rightarrow x
$$

만 수행한다면 어떻게 될까? 이것을 identity function이라고 한다.

$$
f(x)=x
$$

    입력 x -> 새로운 layer -> 그대로 x

이러면 layer를 추가했더라도 최소한 기존 모델의 성능은 유지할 수 있다. 그리고 추가된 layer가 더 좋은 변환을 학습하면 성능이 향상될 수도 있다.

깊은 layer가 identity mapping을 쉽게 학습할 수 있도록 구조를 설계하자가 ResNet의 핵심이다.

## 3. Residual Learning

일반적인 신경망에서는 layer가 원하는 함수 $f(x)$를 직접 학습해야 한다.

    x -> Conv -> Conv -> F(X)

ResNet은 다르게 생각했다. 새로운 함수를 정의한다.

$$
g(x)=f(x)-x
$$

그러면 이렇게 된다.
$$
f(x)=g(x)+x
$$

모델에게 $f(x)$를 전부 새로 만들라고 하지 않고 현재 입력 x에서 얼마나 수정해야 하는지만 학습하라고 하는 것이다.

이때 $g(x)$를 residual, 잔차라고 부른다.

## 4. Residual Connection

ResNet에선 입력 $x$를 convolution layer로 보내는 동시에 출력쪽으로 바로 전달한다.

```text
x ───────────────────┐
│                    │
↓                    │
Conv 3×3             │
↓                    │
BatchNorm            │
↓                    │
ReLU                 │
↓                    │
Conv 3×3             │
↓                    │
BatchNorm            │
│                    │
└──────── + ←────────┘
          ↓
         ReLU
```

Conv를 통과한 결과를 $g(x)$라고 하면 최종 결과는 $y = g(x)+x$ 가 된다.

입력 $x$가 우회해서 직접 전달되는 값을

- Residual Connection
- Shortcut Connection
- Skip Connection

이라고 부른다.

## 5. 왜 이게 더 쉬울까?

예를 들어서 현재 입력을 거의 그대로 유지하는 것이 최선이라고 해보자.

일반 CNN이라면 여러 convolution layer가 직접 $f(x)=x$를 학습해야 한다.

Residual Block에서는 $f(x)=g(x)+x$이다. 우리가 원하는게 $f(x)=x$라면 $g(x)=0$이면 된다. 

`convolution 부분이 아무것도 수정하지 않음`을 학습하면 된다.

일반 네트워크

x -> "처음부터 원하는 결과를 만들어"

ResNet

```text
x ──────────────┐
↓               │
"필요한 것만     │
 수정해"        │
↓               │
g(x) + x ←──────┘
```

이 때문에 매우 깊은 네트워크를 구성하기 훨씬 쉬워졌다.

## 6. Residual Block 구현

In [ ]:
class ResidualBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=stride,
                bias=False
            )

    def forward(self, x):

        y = F.relu(self.bn1(self.conv1(x)))

        y = self.bn2(self.conv2(y))

        y = y + self.shortcut(x) # 여기가 ResNet의 핵심이다.

        return F.relu(y)

## 7. 그런데 x와 g(x)의 크기가 다르면?

덧셈을 하려면 tensor shape이 같아야 한다.

예를 들어서

    g(x) = [32, 128, 28, 28]
    x    = [32, 64, 56, 56]

이라면 g_x + x를 할 수 없다. 채널 수도 다르고 H, W도 다르다. 이럴 때 shortcut쪽에 1x1 Conv를 넣는다.

```text
x
│
├──── 1×1 Conv ─────────┐
│                       │
↓                       │
3×3 Conv                │
↓                       │
3×3 Conv                │
│                       │
└────────── + ←─────────┘
```

[32, 64, 56, 56] -> 1x1 Conv, stride=2 -> [32, 128, 28, 28] 로 변환한다. 그러면 main path의 결과와 shape이 같아져서 더할 수 있다. 

ResNet에서 1x1 Conv는 주로 채널수, H, W 맞추기 역할을 한다.

## 8. ResNet Block에서 BatchNorm

앞 장에서 배운 Batch Normalization이 바로 사용된다.

기존 Residual Block은 대략 이런 구조이다.
```text
Conv  # 특징 추출
 ↓
BatchNorm # activation scale 안정화
 ↓
ReLU # 비선형성
 ↓
Conv
 ↓
BatchNorm
 ↓
+x  # 기존 정보 전달
 ↓
ReLU
```

## 9. ResNet-18 전체 구조

초기 부분은 다음과 비슷하다.

```text
Input
 ↓
7×7 Conv, 64 channels, stride=2
 ↓
BatchNorm
 ↓
ReLU
 ↓
3×3 MaxPool, stride=2
```

그리고 Residual Block을 여러개 사용한다.

```text
64 channels
Residual × 2
 ↓

128 channels
Residual × 2
 ↓

256 channels
Residual × 2
 ↓

512 channels
Residual × 2
 ↓

Global Average Pooling
 ↓
Linear
 ↓
Class Prediction
```

채널은 계속 증가한다. H, W는 계속 감소한다.

## 10. Shape 변화

입력이 [1, 1, 96, 96]이라고 하면 대략 다음처럼 변화한다.

```text
Input [1, 1, 96, 96]
↓
초기 Conv + Pool [1, 64, 24, 24]
↓
Residual Stage 1 [1, 64, 24, 24]
↓
Residual Stage 2 [1, 128, 12, 12]
↓
Residual Stage 3 [1, 256, 6, 6]
↓
Residual Stage 4 [1, 512, 3, 3]
↓
Global Average Pooling [1, 512]
↓
Linear [1, 10]
```

## 11. 왜 Gradient에도 도움이 될까?

ResNet의 중요한 장점 중 하나는 역전파와도 관련되어 있다.

Residual Block이 $y=x+g(x)$라면 이걸 미분하면 
$$
1+
\frac{\partial g(x)}{\partial x}
$$

이렇게 된다. 여기서 중요한 건 1이 존재한다. gradient가 convolution layer만 거쳐야 하는게 아니라 shortcut 경로를 통해서도 전달될 수 있다.

    Loss <- Conv <- Conv <- x
    Loss <- Shortcut <- x

경로가 존재한다. 그래서 매우 깊은 네트워크에서도 gradient가 앞쪽 layer까지 전달되기 쉬워진다.

    ResNet = 단순히 vanishing gradient 해결 기술

보다 더 중요한 설계 철학은

> 기존 mapping을 보존할 수 있는 identity path를 네트워크 안에 명시적으로 만들어 놓은 것이다.

## 12. ResNeXt

ResNet을 더 강하게 만들고 싶으면 여러개의 방법이 있다.

더 깊게 만들기: depth 증가

채널을 더 많이 사용: width 증가

하지만 채널을 많이 증가시키면 convolution 연산량도 크게 증가한다. ResNeXt는 GoogLeNet의 여러 branch 아이디어와 ResNet을 결합한 것에 가깝다.

Grouped Convolution이 핵심이다.

일반 convolution이 모든 채널을 함께 처리한다면

    64 channels -> 하나의 Conv -> 64 channels

Grouped Convloution은 채널을 여러 그룹으로 나눈다.

```text
64 channels
├─ Group 1
├─ Group 2
├─ Group 3
└─ Group 4
각각 Conv
↓
다시 결합
```

그룹 수가 $g$라면 dense convolution에 비해 핵심 연산량과 파라미터를 대량 $1/g$수준으로 줄일 수 있다. 

ResNeXt는 이를 residual block에 사용한다.

## 13. ResNet과 GoogLeNet 비교

GoogLeNet도 branch를 사용했다.

```text
GoogLeNet

         ┌─ 1×1
         ├─ 3×3
입력 ────┼─ 5×5
         └─ Pool
             ↓
           concat

ResNet도 일종의 두 branch 구조로 생각할 수 있다.

ResNet

         ┌─ Conv → Conv ─┐
입력 ────┤               ├─ Add
         └───────────────┘
             Identity
```

하지만 둘의 목적이 다르다.

    GoogLeNet -> 다양한 크기의 특징을 동시에 추출  
    ResNet -> 기존 입력을 보존하며 필요한 변화만 학습

또한 GoogLeNet은 branch 결과를 concat하지만 ResNet은 add한다.

## 14. 오늘의 정리

- 네트워크를 무조건 깊게 만든다고 항상 성능이 좋아지는 것은 아니다.
- ResNet은 새로운 layer가 identity mapping을 쉽게 표현할 수 있도록 설계한다.
- 원하는 mapping을 직접 $f(x)$로 학습하는 대신 residual을 학습한다.
- 입력 $x$를 직접 출력 쪽으로 전달하는 경로를 skip connection 또는 shortcut connection이라고 한다.
- 입력과 출력 shape이 다르면 1×1 Conv로 shape을 맞출 수 있다.
- 기본 ResNet block은 Conv → BN → ReLU → Conv → BN → Add → ReLU 구조를 사용한다.
- ResNet-18은 residual block을 반복적으로 쌓아 구성한다.
- 깊어질수록 H, W는 감소하고 채널 수는 증가한다.
- shortcut은 forward에서 정보를 직접 전달하는 경로를 제공한다.
- backward에서도 gradient가 전달될 수 있는 직접적인 경로를 제공한다.
- ResNet은 매우 깊은 네트워크를 실용적으로 학습할 수 있게 만든 핵심 구조다.
- ResNeXt는 ResNet에 grouped convolution 개념을 추가한다. 